# Sugar Checker — Image Collector

Automated data collection pipeline for Indonesian food and beverage product images.

### Pipeline
```
OpenFoodFacts Parquet
       ↓
Filter Indonesia products → Extract image URLs
       ↓
Download images:
  - Graded products (A-E)   → images/openfoodfacts/
  - Ungraded products       → images/unknown/
       ↓
Web crawling (DDG + Google) → images/crawl/
       ↓
Ready for clustering & training
```

**Run cells in order. Do NOT skip.**

---
## STEP 0 — Mount Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

Mounted at /content/drive
Drive mounted!


---
## STEP 1 — Setup Paths

In [3]:
import os

BASE_DIR     = '/content/drive/MyDrive/sugar_checker_data_V2'
PARQUET_PATH = f'{BASE_DIR}/food.parquet'
JSON_PRODUCTS= f'{BASE_DIR}/indo_products.json'
IMG_DIR      = f'{BASE_DIR}/images'
OFF_DIR      = f'{IMG_DIR}/openfoodfacts'
CRAWL_DIR    = f'{IMG_DIR}/crawl'
UNKNOWN_DIR  = f'{IMG_DIR}/unknown'

for d in [BASE_DIR, IMG_DIR, OFF_DIR, CRAWL_DIR, UNKNOWN_DIR]:
    os.makedirs(d, exist_ok=True)

print('Setup complete!')
print(f'  Base dir             : {BASE_DIR}')
print(f'  Parquet              : {PARQUET_PATH}')
print(f'  Products JSON        : {JSON_PRODUCTS}')
print(f'  OpenFoodFacts images : {OFF_DIR}')
print(f'  Crawl images         : {CRAWL_DIR}')
print(f'  Unknown images       : {UNKNOWN_DIR}')

Setup complete!
  Base dir             : /content/drive/MyDrive/sugar_checker_data_V2
  Parquet              : /content/drive/MyDrive/sugar_checker_data_V2/food.parquet
  Products JSON        : /content/drive/MyDrive/sugar_checker_data_V2/indo_products.json
  OpenFoodFacts images : /content/drive/MyDrive/sugar_checker_data_V2/images/openfoodfacts
  Crawl images         : /content/drive/MyDrive/sugar_checker_data_V2/images/crawl
  Unknown images       : /content/drive/MyDrive/sugar_checker_data_V2/images/unknown


---
## STEP 2 — Install Dependencies

In [4]:
!pip install requests pandas pyarrow icrawler duckduckgo_search -q
print('Dependencies ready!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 45.6 MB/s eta 0:00:0000:0100:01
Dependencies ready!


---
## STEP 3 — Download Parquet

> Automatically skips if file already exists and is valid. Re-downloads if corrupt.

In [5]:
import requests
import os

PARQUET_URL   = 'https://huggingface.co/datasets/openfoodfacts/product-database/resolve/main/food.parquet'
PARQUET_MAGIC = b'PAR1'

def is_valid_parquet(path):
    try:
        if os.path.getsize(path) < 8:
            return False
        with open(path, 'rb') as f:
            header = f.read(4)
            f.seek(-4, 2)
            footer = f.read(4)
        return header == PARQUET_MAGIC and footer == PARQUET_MAGIC
    except:
        return False

def download_parquet(url, dest, max_retries=3):
    for attempt in range(1, max_retries + 1):
        print(f'Attempt {attempt}/{max_retries}...')
        tmp_path = dest + '.tmp'
        try:
            with requests.get(url, stream=True, timeout=60) as r:
                r.raise_for_status()
                total      = int(r.headers.get('content-length', 0))
                downloaded = 0
                with open(tmp_path, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=4 * 1024 * 1024):
                        f.write(chunk)
                        downloaded += len(chunk)
                        if total:
                            print(f'  {downloaded/total*100:.1f}%  ({downloaded/1024/1024:.0f} MB)', end='\r')
            print()
            if is_valid_parquet(tmp_path):
                os.replace(tmp_path, dest)
                print(f'Download OK! ({os.path.getsize(dest)/1024/1024:.1f} MB)')
                return True
            else:
                print(f'  Corrupt file on attempt {attempt}, retrying...')
                os.remove(tmp_path)
        except Exception as e:
            print(f'  Error: {e}')
            if os.path.exists(tmp_path):
                os.remove(tmp_path)
    return False

if os.path.exists(PARQUET_PATH):
    if is_valid_parquet(PARQUET_PATH):
        size_mb = os.path.getsize(PARQUET_PATH) / 1024 / 1024
        print(f'File already exists and valid ({size_mb:.1f} MB), skipping.')
    else:
        print('Existing file is CORRUPT, deleting and re-downloading...')
        os.remove(PARQUET_PATH)
        ok = download_parquet(PARQUET_URL, PARQUET_PATH)
        if not ok:
            raise RuntimeError('Download failed after retries. Check internet / Drive space.')
else:
    ok = download_parquet(PARQUET_URL, PARQUET_PATH)
    if not ok:
        raise RuntimeError('Download failed after retries. Check internet / Drive space.')


Attempt 1/3...
  100.0%  (7001 MB)
Download OK! (7001.1 MB)


---
## STEP 4 — Load Parquet → Extract Image URLs

> Filters Indonesia products only. Extracts image URL per product. No keyword generation.

In [6]:
import pyarrow.parquet as pq
import pandas as pd
import json
import gc

print('Reading parquet (selected columns)...')
table = pq.read_table(
    PARQUET_PATH,
    columns=['code', 'product_name', 'brands', 'countries_tags', 'images', 'nutriscore_grade']
)
print(f'  Total rows global: {len(table):,}')

print('Filtering Indonesia products...')
mask = [
    any(v in ['en:indonesia', 'id:indonesia'] for v in (row.as_py() or []))
    for row in table['countries_tags']
]
indo_table = table.filter(mask)
print(f'  Indonesia products: {len(indo_table):,}')
del table
gc.collect()

indo = indo_table.to_pandas()
del indo_table
gc.collect()

def extract_name(product_name):
    if product_name is None or len(product_name) == 0:
        return None
    for n in product_name:
        if isinstance(n, dict) and n.get('lang') in ('id', 'en'):
            text = n.get('text', '').strip()
            if text:
                return text
    first = product_name[0]
    return first.get('text', '').strip() if isinstance(first, dict) else None

def extract_image_url(code, images):
    if images is None or len(images) == 0 or not code:
        return None
    front_key = None
    first_key = None
    for img in images:
        if not isinstance(img, dict):
            continue
        key   = str(img.get('key', ''))
        imgid = img.get('imgid')
        if 'front' in key and imgid:
            front_key = int(float(imgid))
            break
        if first_key is None and key.isdigit():
            first_key = int(float(key))
    use_key = front_key or first_key
    if not use_key:
        return None
    barcode = str(code).zfill(13)
    path    = '/'.join([barcode[:3], barcode[3:6], barcode[6:9], barcode[9:]])
    return f'https://images.openfoodfacts.org/images/products/{path}/{use_key}.400.jpg'

print('Extracting data...')
indo['name']      = indo['product_name'].apply(extract_name)
indo['image_url'] = indo.apply(
    lambda r: extract_image_url(r['code'], r['images']), axis=1
)

result_df = indo[
    indo['name'].notna() &
    (indo['name'] != '') &
    indo['image_url'].notna()
].copy()

print(f'\nResults:')
print(f'  Products with name & image : {len(result_df):,}')

result = result_df[['code', 'name', 'brands', 'image_url', 'nutriscore_grade']] \
    .to_dict(orient='records')

with open(JSON_PRODUCTS, 'w', encoding='utf-8') as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print(f'Saved: {JSON_PRODUCTS}')
print('\nSample (5):')
for p in result[:5]:
    print(f"  {p['name']} | {p['nutriscore_grade']} | {p['image_url']}")

Reading parquet (selected columns)...
  Total rows global: 4,400,802
Filtering Indonesia products...
  Indonesia products: 7,953
Extracting data...

Results:
  Products with name & image : 6,381
Saved: /content/drive/MyDrive/sugar_checker_data_V2/indo_products.json

Sample (5):
  DeTea | unknown | https://images.openfoodfacts.org/images/products/007/017/705/1174/2.400.jpg
  Japanese green tea | b | https://images.openfoodfacts.org/images/products/007/441/074/1860/33.400.jpg
  Rasa Ayam Bawang | unknown | https://images.openfoodfacts.org/images/products/008/968/601/0015/6.400.jpg
  Indomie Kari Ayam | unknown | https://images.openfoodfacts.org/images/products/008/968/601/0527/4.400.jpg
  Indomie Mi Goreng Original | unknown | https://images.openfoodfacts.org/images/products/008/968/601/0947/17.400.jpg


---
## STEP 5 — Download Images from OpenFoodFacts

> Graded (A/B/C/D/E) → `images/openfoodfacts/` | Ungraded → `images/unknown/`

In [7]:
import requests
import re
import time
import json
from pathlib import Path

HEADERS      = {'User-Agent': 'SugarChecker/1.0 (research project)'}
MAX_DOWNLOAD = 4000

Path(OFF_DIR).mkdir(parents=True, exist_ok=True)
Path(UNKNOWN_DIR).mkdir(parents=True, exist_ok=True)

with open(JSON_PRODUCTS, 'r', encoding='utf-8') as f:
    result = json.load(f)

downloaded = 0
skipped    = 0
errors     = 0

for p in result:
    if downloaded >= MAX_DOWNLOAD:
        break

    image_url = p.get('image_url')
    if not image_url:
        skipped += 1
        continue

    name      = p.get('name', 'unknown')
    grade     = p.get('nutriscore_grade', '').upper()
    safe_name = re.sub(r'[^a-zA-Z0-9]', '_', name)[:50]

    # Route by grade
    if grade in ['A', 'B', 'C', 'D', 'E']:
        img_path = Path(OFF_DIR) / f'off_{safe_name}_{downloaded:04d}.jpg'
    else:
        img_path = Path(UNKNOWN_DIR) / f'off_{safe_name}_{downloaded:04d}.jpg'

    if img_path.exists():
        downloaded += 1
        continue

    try:
        r = requests.get(image_url, headers=HEADERS, timeout=10)
        if r.status_code == 200 and len(r.content) > 1000:
            img_path.write_bytes(r.content)
            downloaded += 1
        else:
            skipped += 1
        time.sleep(0.3)
    except:
        errors += 1

    if downloaded % 100 == 0 and downloaded > 0:
        print(f'Progress: {downloaded} downloaded')

print(f'\nOpenFoodFacts download complete!')
print(f'  Downloaded : {downloaded}')
print(f'  Skipped    : {skipped}')
print(f'  Errors     : {errors}')


OpenFoodFacts download complete!
  Downloaded : 4000
  Skipped    : 1
  Errors     : 0


---
## STEP 6 — Crawl Images (DDG + Google)

> All crawled images go directly to `images/crawl/`. No keyword merging.

In [8]:
import sys
sys.stderr = open('/dev/null', 'w')

from duckduckgo_search import DDGS
from icrawler.builtin import GoogleImageCrawler
import requests
from pathlib import Path
import warnings
import logging
import time

warnings.filterwarnings('ignore')
logging.disable(logging.CRITICAL)

Path(CRAWL_DIR).mkdir(parents=True, exist_ok=True)

CATEGORIES = [
    # Food
    'kemasan makanan indonesia',
    'snack indonesia kemasan',
    'mie instan kemasan indonesia',
    'biskuit kemasan indonesia',
    'makanan ringan kemasan indonesia',
    'coklat kemasan indonesia',
    'sereal kemasan indonesia',
    # Drink
    'kemasan minuman indonesia',
    'minuman botol indonesia',
    'minuman sachet indonesia',
    'minuman kaleng indonesia',
    'teh botol kemasan indonesia',
    'susu kemasan indonesia',
    'jus kemasan indonesia',
]

MAX_PER_CATEGORY = 100
HEADERS_DL       = {'User-Agent': 'Mozilla/5.0'}
total_downloaded = 0

for category in CATEGORIES:
    print(f'\nCrawling: {category}...')
    ddg_dl  = 0
    goog_dl = 0

    # 1 — DDG
    try:
        with DDGS() as ddgs:
            results = ddgs.images(category, max_results=MAX_PER_CATEGORY)
            for r in results:
                url = r.get('image')
                if url:
                    try:
                        resp = requests.get(url, headers=HEADERS_DL, timeout=8)
                        if resp.status_code == 200 and len(resp.content) > 1000:
                            fname = Path(CRAWL_DIR) / f"ddg_{category.replace(' ', '_')}_{ddg_dl:04d}.jpg"
                            fname.write_bytes(resp.content)
                            ddg_dl += 1
                    except:
                        pass
        print(f'  [DDG]    {ddg_dl} images')
    except Exception as e:
        print(f'  [DDG ERROR] {e}')

    time.sleep(2)

    # 2 — Google
    try:
        crawler = GoogleImageCrawler(
            storage            = {'root_dir': CRAWL_DIR},
            feeder_threads     = 1,
            parser_threads     = 1,
            downloader_threads = 4,
        )
        before = len(list(Path(CRAWL_DIR).glob('*.*')))
        crawler.crawl(keyword=category, max_num=MAX_PER_CATEGORY, file_idx_offset='auto')
        after   = len(list(Path(CRAWL_DIR).glob('*.*')))
        goog_dl = after - before
        print(f'  [Google] {goog_dl} images')
    except Exception as e:
        print(f'  [Google ERROR] {e}')

    total_downloaded += ddg_dl + goog_dl
    print(f'  [OK] {ddg_dl + goog_dl} images')

print(f'\nCrawl done!')
print(f'  Total crawl images : {total_downloaded}')



Crawling: kemasan makanan indonesia...
  [DDG]    97 images
  [Google] 0 images
  [OK] 97 images

Crawling: snack indonesia kemasan...
  [DDG ERROR] https://duckduckgo.com/i.js?o=json&q=snack+indonesia+kemasan&l=us-en&vqd=4-164381019546753811997692503810771989726&p=1&f=%2C%2C%2C%2C%2C 403 Ratelimit
  [Google] 0 images
  [OK] 0 images

Crawling: mie instan kemasan indonesia...
  [DDG ERROR] https://duckduckgo.com/i.js?o=json&q=mie+instan+kemasan+indonesia&l=us-en&vqd=4-288394296004988299950543130694522017203&p=1&f=%2C%2C%2C%2C%2C 403 Ratelimit
  [Google] 0 images
  [OK] 0 images

Crawling: biskuit kemasan indonesia...
  [DDG ERROR] https://duckduckgo.com/i.js?o=json&q=biskuit+kemasan+indonesia&l=us-en&vqd=4-43634787929022581949482986499655184924&p=1&f=%2C%2C%2C%2C%2C 403 Ratelimit
  [Google] 0 images
  [OK] 0 images

Crawling: makanan ringan kemasan indonesia...
  [DDG ERROR] https://duckduckgo.com/i.js?o=json&q=makanan+ringan+kemasan+indonesia&l=us-en&vqd=4-319484306937235270615972587

---
## STEP 7 — Final Dataset Summary

In [9]:
from pathlib import Path

off_count     = len(list(Path(OFF_DIR).glob('*.*')))
crawl_count   = len(list(Path(CRAWL_DIR).glob('*.*')))
unknown_count = len(list(Path(UNKNOWN_DIR).glob('*.*')))

print('Dataset Summary:')
print('=' * 55)
print(f'  images/openfoodfacts/ : {off_count} images (graded, from OFF)')
print(f'  images/crawl/         : {crawl_count} images (DDG + Google)')
print(f'  images/unknown/       : {unknown_count} images (no grade)')
print(f'  Total                 : {off_count + crawl_count + unknown_count} images')
print('=' * 55)
print(f'\nReady for clustering and training!')

Dataset Summary:
  images/openfoodfacts/ : 330 images (graded, from OFF)
  images/crawl/         : 97 images (DDG + Google)
  images/unknown/       : 3670 images (no grade)
  Total                 : 4097 images

Ready for clustering and training!
